# Seed Ablation

Runs multiple random seeds for selected architectures. The base config exposes optional force loss and Hessian regularization.

In [ ]:
import os
import jax.numpy as jnp

from properties import SlinkyN3Properties
from run_architectures import SweepConfig, subset_energy_only, subset_main_paper_candidates, subset_all
from seed_ablation_utils import (
    run_seed_ablation,
    print_seed_summary,
    save_seed_summary_json,
    make_seed_ablation_plots,
)


In [ ]:
# Dataset / physical setup
properties = SlinkyN3Properties(mass=0.3)
train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset.npz"

selected_architectures = subset_energy_only()
# selected_architectures = subset_main_paper_candidates()
# selected_architectures = subset_all()

# Optional force loss. Leave strength at 0.0 for displacement-only training.
force_loss_strength = 1.0
force_key = None
force_components = (0,)
force_sign = 1.0
return_loss_components = force_loss_strength != 0.0

# Optional Hessian regularizer.
hessian_reg_strength = 0.0
hessian_reg_probes = 1
hessian_reg_seed = 0


In [ ]:
base_cfg = SweepConfig(
    output_dir="seed_ablation_outputs_n3_slinky_simdata_force_optional",
    n_epochs=500,
    lr=1e-2,
    seed=0,
    seed_list=tuple(range(25)),
    hidden=(10,),
    input_mode="invariant",
    activation="tanh",
    corr_factor=0.01,
    only_stretching_NN=False,
    zero_reference=True,
    valid_every=1,
    max_dlambda=5e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=hessian_reg_strength,
    hessian_reg_probes=hessian_reg_probes,
    hessian_reg_seed=hessian_reg_seed,
    force_key=force_key,
    force_loss_strength=force_loss_strength,
    force_components=force_components,
    force_sign=force_sign,
    return_loss_components=return_loss_components,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_energy_landscapes=True,
    verbose=True,
)


In [ ]:
all_seed_results = run_seed_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    base_cfg=base_cfg,
    selected_architectures=selected_architectures,
)

print_seed_summary(all_seed_results)


In [ ]:
summary_dir = os.path.join(base_cfg.output_dir, "seed_ablation_summary")
save_seed_summary_json(all_seed_results, output_dir=summary_dir)
make_seed_ablation_plots(all_seed_results, output_dir=summary_dir, traj_idx=0)
summary_dir
